In [1]:
import pandas as pd
import geopandas as gpd
import os

In [2]:
#load files 

MW_SHP = "/home/i9i/Documents/Phase_2/gadm41_MWI_shp/gadm41_MWI_1.shp"
DISTRICT_RISK = "/home/i9i/Documents/Phase_2/From_Phase_1/malawi_district_risk.csv"

dr = pd.read_csv(DISTRICT_RISK)
sh = gpd.read_file(MW_SHP)


print(f'Shapefile polygons : {len(sh)}')
print(f'CSV districts      : {dr["district"].nunique()}')

# ── Fix city names in CSV ─────────────────────────────────────────────────────
city_map = {
    'Blantyre City' : 'Blantyre',
    'Lilongwe City' : 'Lilongwe',
    'Mzuzu City'    : 'Mzimba'
}
dr['district'] = dr['district'].replace(city_map)

# ── Aggregate — sum counts, average rates ─────────────────────────────────────
dr_agg = dr.groupby('district', as_index=False).agg(
    region         = ('region',         'first'),
    total_records  = ('total_records',  'sum'),
    total_spikes   = ('total_spikes',   'sum'),
    critical_count = ('critical_count', 'sum'),
    severe_count   = ('severe_count',   'sum'),
    moderate_count = ('moderate_count', 'sum'),
    avg_price      = ('avg_price',      'mean'),
    spike_rate_pct = ('spike_rate_pct', 'mean'),
    risk_score     = ('risk_score',     'sum'),
)


Shapefile polygons : 28
CSV districts      : 30


In [18]:
# print(f'Polygons : {len(sh)}')
# print(f'Columns  : {sh.columns.tolist()}')
# print(f'\nSample rows:')
# print(sh[[c for c in sh.columns if 'NAME' in c]].head(10))


# print('=== SHAPEFILE DISTRICT NAMES ===')
# shapefile_names = sorted(sh['NAME_1'].unique())
# print(shapefile_names)

# print('\n=== YOUR CSV DISTRICT NAMES ===')
# csv_names = sorted(dr['district'].unique())
# print(csv_names)

# print("\n=======CHECK MISMATCH=====")
# in_shapefile_not_csv = set(shapefile_names) - set(csv_names)
# in_csv_not_shapefile = set(csv_names) - set(shapefile_names)

# print(f"\nNames in Shapeflie but not in CSV: {in_shafile_not_csv}")
# print(f"\nNames in CSV but not in shapefile: {in_csv_not_shafile}")

In [4]:
# ── Add Likoma with zeros ─────────────────────────────────────────────────────
likoma = {col: 0 for col in dr_agg.columns}
likoma['district'] = 'Likoma'
likoma['region']   = 'Northern Region'
dr_agg = pd.concat([dr_agg, pd.DataFrame([likoma])], ignore_index=True)

# ── Verify names match ────────────────────────────────────────────────────────
shapefile_names = set(sh['NAME_1'])
csv_names       = set(dr_agg['district'])

print(f'\nIn shapefile not CSV : {shapefile_names - csv_names}')
print(f'In CSV not shapefile : {csv_names - shapefile_names}')



In shapefile not CSV : set()
In CSV not shapefile : set()


In [5]:
# ── Merge on NAME_1 ───────────────────────────────────────────────────────────
merged = sh.merge(
    dr_agg,
    left_on  = 'NAME_1',     # district name in shapefile
    right_on = 'district',   # district name in CSV
    how      = 'left'
)

merged = merged.fillna(0)

print(f'\nMerged rows : {len(merged)}')
print(f'\nTop 10 districts by risk score:')
print(merged[['NAME_1','risk_score','critical_count','severe_count']]
      .sort_values('risk_score', ascending=False)
      .head(10)
      .to_string(index=False))



Merged rows : 28

Top 10 districts by risk score:
  NAME_1  risk_score  critical_count  severe_count
Machinga         436              25            79
 Mulanje         277              13            65
   Zomba         262              21            54
  Nsanje         252              11            41
Chikwawa         217               9            47
    Dowa         199               9            34
  Ntcheu         185               9            45
Blantyre         181               8            40
 Ntchisi         175              11            36
Phalombe         167               8            37


In [6]:

# ── Save ──────────────────────────────────────────────────────────────────────
dr_agg.to_csv('malawi_district_risk_prepared.csv', index=False)
print('\nSaved: malawi_district_risk_prepared.csv')

merged.to_file('malawi_districts_risk.geojson', driver='GeoJSON')
print('Saved: malawi_districts_risk.geojson')


Saved: malawi_district_risk_prepared.csv
Saved: malawi_districts_risk.geojson
